

---
1. Environment Setup and Imports


In [1]:
# Install required libraries
!pip install gradio

# Imports
import sqlite3
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
import gradio as gr


2. Database Initialization

In [2]:
# Initialize SQLite database and tables
conn = sqlite3.connect('students.db')
cursor = conn.cursor()

# Create users table (if not exists)
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    email TEXT UNIQUE,
    skills TEXT
)
""")

# Create messages table (if not exists)
cursor.execute("""
CREATE TABLE IF NOT EXISTS messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sender_id INTEGER,
    receiver_id INTEGER,
    message TEXT,
    timestamp TEXT,
    FOREIGN KEY(sender_id) REFERENCES users(id),
    FOREIGN KEY(receiver_id) REFERENCES users(id)
)
""")

# Populate sample user data only if table is empty
cursor.execute("SELECT COUNT(*) FROM users")
user_count = cursor.fetchone()[0]
if user_count == 0:
    names = [
        "Alice Johnson", "Bob Smith", "Charlie Lee", "Diana Green", "Ethan Brown",
        "Fiona White", "George Black", "Hannah Scott", "Ian Clark", "Julia Adams",
        "Kevin Baker", "Laura Carter", "Michael Davis", "Nina Evans", "Oscar Fisher",
        "Patricia Garcia", "Quentin Hall", "Rachel Irvine", "Samuel Jones", "Tina Knight",
        "Umar Latif", "Victoria Miller", "Walter Nelson", "Xavier Owens", "Yvonne Perez",
        "Zachary Quinn", "Aaron Rogers", "Bethany Stewart", "Carlos Taylor", "Diana Underwood",
        "Eliot Vance", "Felicity Walker", "Gavin Xu", "Hanna Young", "Isaac Zimmerman",
        "Jackie Allen", "Kurtis Bryant", "Linda Collins", "Mario Diaz", "Nora Ellis",
        "Oscar Franklin", "Paula Gomez", "Quincy Harris", "Rita Ivers", "Steve Jackson",
        "Tracy Kim", "Uma Levin", "Victor Manning", "Wendy Norris", "Xander O'Brien"
    ]

    skills_pool = [
        "Python", "Data Science", "Machine Learning", "HTML/CSS", "JavaScript",
        "C++", "Matlab", "Photography", "Graphic Design", "Spanish", "French",
        "Basketball", "Soccer", "Painting", "Guitar", "Piano", "Chemistry",
        "Biology", "Robotics", "Public Speaking", "Cooking", "Singing",
        "Dancing", "Yoga", "Physics", "Mathematics", "Statistics", "Chess",
        "Writing", "Video Editing"
    ]

    import random
    random.seed(42)
    for name in names:
        email = name.lower().replace(" ", ".") + "@example.com"
        num_skills = random.randint(3, 5)
        skills = random.sample(skills_pool, num_skills)
        skills_str = ", ".join(skills)
        cursor.execute("INSERT OR IGNORE INTO users (name, email, skills) VALUES (?, ?, ?)",
                       (name, email, skills_str))
    conn.commit()

conn.close()


3. Machine Learning: SkillMatcher Class

In [3]:
class SkillMatcher:
    def __init__(self, db_path='students.db'):
        # Load user profiles and skills from database
        conn = sqlite3.connect(db_path)
        df = pd.read_sql("SELECT id, skills FROM users", conn)
        conn.close()

        self.user_ids = df['id'].tolist()
        # Prepare skill documents (combine skills)
        self.skill_docs = df['skills'].fillna("").apply(lambda x: x.replace(",", " ")).tolist()
        # Store skill sets for evaluation (splitting on comma)
        self.skill_sets = [set(s.strip() for s in skills.split(",")) for skills in df['skills']]

        # Fit TF-IDF on skill documents
        self.vectorizer = TfidfVectorizer()
        if self.skill_docs:
            self.tfidf_matrix = self.vectorizer.fit_transform(self.skill_docs)
        else:
            self.tfidf_matrix = None

        # Compute Top-K accuracy (Top-3 by default)
        self.topk = 3
        self.accuracy = self.evaluate_topk(self.topk)

    def evaluate_topk(self, k=3):
        if self.tfidf_matrix is None or not self.user_ids:
            return 0.0
        sims = cosine_similarity(self.tfidf_matrix)
        n = len(self.user_ids)
        hits = 0
        for i in range(n):
            sim_scores = list(sims[i])
            # Exclude self-match
            sim_scores[i] = -1
            # Indices of top-k similar users
            topk_idx = np.argsort(sim_scores)[-k:][::-1]
            # Check if any top-k share skills
            found = False
            for j in topk_idx:
                if self.skill_sets[i].intersection(self.skill_sets[j]):
                    found = True
                    break
            if found:
                hits += 1
        accuracy = (hits / n) * 100
        return round(accuracy, 2)

    def find_matches(self, input_skills, top_k=5):
        input_str = input_skills.strip()
        if not input_str or self.tfidf_matrix is None:
            return []
        query_vec = self.vectorizer.transform([input_str.replace(",", " ")])
        sims = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_k_indices = np.argsort(sims)[-top_k:][::-1]
        matches = []
        for idx in top_k_indices:
            score = sims[idx]
            user_id = self.user_ids[idx]
            matches.append((user_id, round(float(score*100), 2)))
        return matches


4. Utility Functions

In [4]:
DB_PATH = 'students.db'

def mask_id(user_id):
    return f"User_{user_id:03d}"

def add_profile(name, email, skills):
    if not name or not email or not skills:
        return "Please provide name, email, and at least one skill."
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    # Check if user already exists by email
    cursor.execute("SELECT id FROM users WHERE email = ?", (email,))
    result = cursor.fetchone()
    if result:
        user_id = result[0]
        conn.close()
        return f"Email already exists. Your User ID is {mask_id(user_id)}."
    # Insert new profile
    cursor.execute("INSERT INTO users (name, email, skills) VALUES (?, ?, ?)",
                   (name, email, skills))
    conn.commit()
    user_id = cursor.lastrowid
    conn.close()
    return f"Profile added. Your User ID is {mask_id(user_id)}."

def search_by_skill(skill):
    skill_search = skill.strip().lower()
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT id, skills FROM users")
    rows = cursor.fetchall()
    conn.close()
    results = []
    for user_id, skills in rows:
        if skill_search in skills.lower():
            results.append([mask_id(user_id), skills])
    df = pd.DataFrame(results, columns=["UserID", "Teach Skills"])
    return df

def find_ai_matches(input_skills):
    if not input_skills:
        return pd.DataFrame([], columns=["UserID", "Confidence Score (%)"])
    # Retrain model to include new profiles
    matcher = SkillMatcher(DB_PATH)
    matches = matcher.find_matches(input_skills, top_k=5)
    data = []
    for user_id, score in matches:
        data.append([mask_id(user_id), score])
    df = pd.DataFrame(data, columns=["UserID", "Confidence Score (%)"])
    return df

def get_conversation(user_id, friend_id):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    query = """
        SELECT sender_id, receiver_id, message, timestamp FROM messages
        WHERE (sender_id=? AND receiver_id=?) OR (sender_id=? AND receiver_id=?)
        ORDER BY id
    """
    cursor.execute(query, (user_id, friend_id, friend_id, user_id))
    rows = cursor.fetchall()
    conn.close()
    data = []
    for sender, receiver, message, timestamp in rows:
        data.append([mask_id(sender), message])
    df = pd.DataFrame(data, columns=["UserID", "Message"])
    return df

def send_message(user_mask, friend_mask, message):
    if not message or not user_mask or not friend_mask:
        return get_conversation(int(user_mask.split("_")[1]), int(friend_mask.split("_")[1])), ""
    user_id = int(user_mask.split("_")[1])
    friend_id = int(friend_mask.split("_")[1])
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    timestamp = datetime.now().isoformat()
    cursor.execute(
        "INSERT INTO messages (sender_id, receiver_id, message, timestamp) VALUES (?, ?, ?, ?)",
        (user_id, friend_id, message, timestamp)
    )
    conn.commit()
    conn.close()
    convo_df = get_conversation(user_id, friend_id)
    return convo_df, ""

def dashboard_stats():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    # Total users
    cursor.execute("SELECT COUNT(*) FROM users")
    total_users = cursor.fetchone()[0]
    # Total unique skills
    cursor.execute("SELECT skills FROM users")
    rows = cursor.fetchall()
    conn.close()
    skill_set = set()
    for (skills,) in rows:
        for skill in skills.split(","):
            skill_set.add(skill.strip().lower())
    total_skills = len(skill_set)
    # Model Top-3 accuracy
    matcher = SkillMatcher(DB_PATH)
    top3_accuracy = matcher.accuracy
    data = [
        ["Total Users", total_users],
        ["Total Unique Skills", total_skills],
        [f"Top-{matcher.topk} Accuracy (%)", top3_accuracy]
    ]
    df = pd.DataFrame(data, columns=["Metric", "Value"])
    return df


5. Gradio Interface

In [5]:
with gr.Blocks(title="Student Skill Exchange Platform using Machine Learning") as demo:

    gr.Markdown("## Student Skill Exchange Platform using Machine Learning")

    with gr.Tabs():

        # ================= PROFILE =================
        with gr.Tab("Profile"):

            name = gr.Textbox(label="Name")
            email = gr.Textbox(label="Email")
            skills = gr.Textbox(label="Skills (comma separated)")
            add_btn = gr.Button("Create Profile")
            profile_status = gr.Textbox(label="Status", interactive=False)

            add_btn.click(add_profile,
                          inputs=[name, email, skills],
                          outputs=profile_status)

        # ================= DISCOVER =================
        with gr.Tab("Discover"):

            search_box = gr.Textbox(label="Search by Skill")
            search_btn = gr.Button("Search")
            discover_output = gr.Dataframe(
                headers=["UserID", "Teach Skills"],
                datatype=["str", "str"],
                interactive=False
            )

            search_btn.click(search_by_skill,
                             inputs=search_box,
                             outputs=discover_output)

        # ================= AI MATCH =================
        with gr.Tab("AI Match"):

            skill_input = gr.Textbox(label="Enter Your Skills")
            match_btn = gr.Button("Find Matches")
            match_output = gr.Dataframe(
                headers=["UserID", "Confidence Score (%)"],
                datatype=["str", "number"],
                interactive=False
            )

            match_btn.click(find_ai_matches,
                            inputs=skill_input,
                            outputs=match_output)

        # ================= CHAT =================
        with gr.Tab("Chat"):

            gr.Markdown("### Secure Messaging")

            def get_user_ids():
                conn = sqlite3.connect(DB_PATH)
                cursor = conn.cursor()
                cursor.execute("SELECT id FROM users")
                ids = [mask_id(row[0]) for row in cursor.fetchall()]
                conn.close()
                return ids

            user_dropdown = gr.Dropdown(label="Your UserID")
            friend_dropdown = gr.Dropdown(label="Chat With UserID")
            refresh_users_btn = gr.Button("Refresh Users")

            chat_output = gr.Dataframe(
                headers=["UserID", "Message"],
                datatype=["str", "str"],
                interactive=False
            )

            message_input = gr.Textbox(label="Message")
            send_button = gr.Button("Send")

            # Refresh dropdown users
            def refresh_users():
                ids = get_user_ids()
                return gr.update(choices=ids), gr.update(choices=ids)

            refresh_users_btn.click(
                refresh_users,
                outputs=[user_dropdown, friend_dropdown]
            )

            # Load conversation
            def load_chat(user_mask, friend_mask):
                if not user_mask or not friend_mask:
                    return pd.DataFrame(columns=["UserID", "Message"])

                user_id = int(user_mask.split("_")[1])
                friend_id = int(friend_mask.split("_")[1])

                return get_conversation(user_id, friend_id)

            user_dropdown.change(
                load_chat,
                inputs=[user_dropdown, friend_dropdown],
                outputs=chat_output
            )

            friend_dropdown.change(
                load_chat,
                inputs=[user_dropdown, friend_dropdown],
                outputs=chat_output
            )

            # Send and refresh
            def send_and_refresh(user_mask, friend_mask, msg):
                if not user_mask or not friend_mask or not msg:
                    return pd.DataFrame(columns=["UserID", "Message"]), ""

                user_id = int(user_mask.split("_")[1])
                friend_id = int(friend_mask.split("_")[1])

                conn = sqlite3.connect(DB_PATH)
                cursor = conn.cursor()
                timestamp = datetime.now().isoformat()

                cursor.execute(
                    "INSERT INTO messages (sender_id, receiver_id, message, timestamp) VALUES (?, ?, ?, ?)",
                    (user_id, friend_id, msg, timestamp)
                )
                conn.commit()
                conn.close()

                updated_chat = get_conversation(user_id, friend_id)
                return updated_chat, ""

            send_button.click(
                send_and_refresh,
                inputs=[user_dropdown, friend_dropdown, message_input],
                outputs=[chat_output, message_input]
            )

        # ================= DASHBOARD =================
        with gr.Tab("Dashboard"):

            dash_btn = gr.Button("Refresh Statistics")

            dash_output = gr.Dataframe(
                headers=["Metric", "Value"],
                datatype=["str", "str"],
                interactive=False
            )

            dash_btn.click(dashboard_stats,
                           outputs=dash_output)

    demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://27cca4494db5c2a50c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
